In [1]:
"""
Dense Retrieval RAG Pipeline -- Production-Grade
=================================================
Architecture   : FIVE retrieval configurations compared side by side:
                 (1) Sparse BM25              -- lexical baseline (TF-IDF style)
                 (2) Dense Bi-Encoder         -- FAISS + BGE embeddings
                 (3) Dense + Cross-Encoder    -- Bi-Encoder candidate retrieval
                                                 followed by Cross-Encoder reranking
                 (4) Hybrid (BM25 + Dense)    -- EnsembleRetriever + RRF fusion
                 (5) Hybrid + Cross-Encoder   -- Full production stack (best quality)

Bi-Encoder     : BAAI/bge-large-en-v1.5  (1024-dim, MTEB SOTA, query instruction prefix)
Cross-Encoder  : BAAI/bge-reranker-base  (BERT-based, trained on MS-MARCO)
Sparse Index   : BM25Retriever  (langchain-community, rank-bm25 backend)
Dense Index    : FAISS  (facebook/faiss, IndexFlatIP for inner product cosine)
Fusion         : EnsembleRetriever  (Reciprocal Rank Fusion, RRF)
Reranker       : ContextualCompressionRetriever + CrossEncoderReranker
LLM            : Ollama  (ChatOllama -- qwen2.5-coder:7b, local inference)
Data Source    : Three publicly available research PDFs from arXiv

Reference PDFs:
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf

    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf

    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
Core Concept -- WHAT IS DENSE RETRIEVAL AND WHY IT EXISTS
=============================================================================
Traditional information retrieval is SPARSE: it represents both queries and
documents as high-dimensional but sparse vectors where each dimension is a
vocabulary token. BM25 (the industry standard for keyword search) is the
canonical sparse method. It excels at exact keyword matching but fundamentally
cannot handle vocabulary mismatch: if the document says "autonomous vehicle"
and the query says "self-driving car", BM25 scores the match as zero because
no token overlaps.

Dense Retrieval encodes both queries and documents into LOW-DIMENSIONAL DENSE
vectors (e.g., 1024 real-valued dimensions) using a neural bi-encoder model.
Similarity is measured as cosine or inner-product distance in this continuous
embedding space. Because the embedding model has learned semantic relationships
during training, "autonomous vehicle" and "self-driving car" produce nearby
vectors and score high regardless of token overlap.

The Bi-Encoder / Cross-Encoder distinction is the architectural heart of
production dense retrieval:

    BI-ENCODER (retrieval stage, fast):
        Encodes query and each document INDEPENDENTLY into fixed-size vectors.
        Similarity = dot_product(query_vec, doc_vec).
        Pre-compute all document vectors offline and index them in FAISS.
        Query time: ONE embedding call + ONE ANN search. Millisecond latency.
        Limitation: because query and document are encoded independently,
        the model cannot attend to token-level interactions between them.
        This causes semantic drift at the margin -- the 4th and 5th most
        similar documents can be topically irrelevant while scoring close
        to the top-3.

    CROSS-ENCODER (reranking stage, precise):
        Encodes query and document JOINTLY: both are concatenated and fed
        through the transformer as a single sequence.
        Output: a scalar relevance score (not an embedding).
        The model can attend to every token in the query while processing
        every token in the document simultaneously.
        This produces far more precise relevance judgments than the dot
        product of independently-computed embeddings.
        Limitation: O(N * model_forward_passes) at inference time where N
        is the candidate set size. Cannot be pre-computed offline.
        Used ONLY on the top-K bi-encoder candidates (typically K=20-50),
        not on the entire corpus.

    PRODUCTION PIPELINE (this code):
        1. BM25 retrieves top-20 lexical candidates (fast, low-cost).
        2. Dense Bi-Encoder retrieves top-20 semantic candidates (fast).
        3. RRF merges and deduplicates both lists --> top-20 fused candidates.
        4. Cross-Encoder rescores all 20 --> returns top-4 highest-quality.
        5. Top-4 context chunks go to the LLM for answer generation.

=============================================================================
Why BAAI/bge-large-en-v1.5 as the Bi-Encoder
=============================================================================
BGE-large-en-v1.5 (BAAI General Embedding) holds top-tier MTEB English
retrieval benchmark scores among open-source models. Key properties:
    - 1024-dimensional embeddings (vs. 384 for all-MiniLM-L6-v2)
    - Trained with contrastive learning on 1B+ pairs
    - Requires a query instruction prefix for retrieval tasks:
      "Represent this sentence for searching relevant passages: {query}"
    - Documents do NOT get an instruction prefix (asymmetric encoding)
The instruction prefix is the most commonly missed configuration detail
for BGE models. Without it, retrieval quality degrades measurably.

=============================================================================
Why BAAI/bge-reranker-base as the Cross-Encoder
=============================================================================
BGE-reranker-base is a BERT-base architecture fine-tuned specifically on
MS-MARCO passage ranking with hard negative mining. It produces a logit
score (not normalized) for each (query, passage) pair. Compared to
ms-marco-MiniLM-L-6-v2 (the other common choice), bge-reranker-base:
    - Is larger and more accurate (BERT-base vs. 6-layer MiniLM)
    - Is from the same BGE family as the bi-encoder (consistent training data)
    - Has better recall on academic/technical text (less commercial web bias)

=============================================================================
Why Ollama + qwen2.5-coder:7b as the LLM
=============================================================================
Ollama provides a local inference server that runs open-source LLMs without
any API keys or cloud costs. qwen2.5-coder:7b is a 7-billion parameter
code-and-reasoning model from Alibaba's Qwen2.5 series:
    - Strong performance on technical and research Q&A tasks
    - 7B params fits comfortably in 8GB VRAM or can run on CPU
    - Local inference: no data leaves the machine (privacy-preserving)
    - No rate limits, no API costs
Prerequisite: Ollama must be running locally (ollama serve) and the model
must be pulled: ollama pull qwen2.5-coder:7b
"""


'\nDense Retrieval RAG Pipeline -- Production-Grade\n=================================================\nArchitecture   : FIVE retrieval configurations compared side by side:\n                 (1) Sparse BM25              -- lexical baseline (TF-IDF style)\n                 (2) Dense Bi-Encoder         -- FAISS + BGE embeddings\n                 (3) Dense + Cross-Encoder    -- Bi-Encoder candidate retrieval\n                                                 followed by Cross-Encoder reranking\n                 (4) Hybrid (BM25 + Dense)    -- EnsembleRetriever + RRF fusion\n                 (5) Hybrid + Cross-Encoder   -- Full production stack (best quality)\n\nBi-Encoder     : BAAI/bge-large-en-v1.5  (1024-dim, MTEB SOTA, query instruction prefix)\nCross-Encoder  : BAAI/bge-reranker-base  (BERT-based, trained on MS-MARCO)\nSparse Index   : BM25Retriever  (langchain-community, rank-bm25 backend)\nDense Index    : FAISS  (facebook/faiss, IndexFlatIP for inner product cosine)\nFusion       

In [2]:
# pip install -U \
# langchain \
# langchain-core \
# langchain-community \
# langchain-ollama \
# langchain-huggingface \
# faiss-cpu \
# sentence-transformers \
# rank-bm25 \
# pypdf
#
# Also ensure Ollama is installed and the model is pulled:
#   ollama pull qwen2.5-coder:7b


In [3]:
# !pip install langchain-ollama

In [4]:
# ============================================================================
# STANDARD LIBRARIES
# ============================================================================

import os
import sys
import time
import logging
import urllib.request

from pathlib import Path
from typing import List, Dict, Tuple, Optional

# ============================================================================
# LANGCHAIN CORE
# ============================================================================

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================================
# TEXT SPLITTING
# ============================================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================================
# DOCUMENT LOADERS
# ============================================================================

from langchain_community.document_loaders import PyPDFLoader

# ============================================================================
# VECTOR STORE
# ============================================================================

from langchain_community.vectorstores import FAISS

# ============================================================================
# EMBEDDINGS
# ============================================================================

from langchain_huggingface import HuggingFaceEmbeddings

# ============================================================================
# RETRIEVERS
# ============================================================================

from langchain_community.retrievers import BM25Retriever

from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)

# ============================================================================
# RERANKERS / CROSS ENCODERS
# ============================================================================

from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

# ============================================================================
# PROMPTS
# ============================================================================

from langchain_core.prompts import ChatPromptTemplate

# ============================================================================
# OLLAMA LLM
# ============================================================================

from langchain_ollama import ChatOllama


C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0513 22:17:36.121000 38164 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [5]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("dense_retrieval_rag")


In [6]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class DenseRAGConfig:
    """
    Centralized configuration for the Dense Retrieval RAG pipeline.

    Retrieval Topology Rationale:
        BIENCODER_CANDIDATE_K = 20:
            The bi-encoder's precision drops after the top-5 or so results.
            Fetching 20 candidates ensures the truly relevant documents are
            present in the candidate pool for the cross-encoder to find.
            Empirically, recall@20 for BGE-large is ~95% on BEIR benchmarks,
            meaning 95% of the time the correct document is in the top 20.

        BM25_CANDIDATE_K = 20:
            Same rationale. BM25 complements dense retrieval by finding
            exact keyword matches that the embedding model might not surface
            (proper nouns, technical abbreviations, newly coined terms).

        RERANKER_TOP_N = 4:
            The cross-encoder re-scores all 20 (or 40 in hybrid) candidates
            and returns the top 4. This is the number of chunks the LLM
            actually sees. 4 * ~512 chars = ~2048 chars of context, fitting
            comfortably in a single prompt with room for the question
            and instruction.

        ENSEMBLE_WEIGHTS = [0.4, 0.6]:
            BM25 and dense retriever weights in the EnsembleRetriever.
            Dense gets slightly higher weight (0.6) because for research paper
            Q&A, semantic matching dominates over keyword matching.
            For legal/regulatory text with exact term requirements, use [0.6, 0.4].

    BGE Query Instruction:
        BGE models require an instruction prefix ONLY for queries, NOT for
        documents. The instruction tells the model that this text is a
        retrieval query, activating query-specific attention patterns learned
        during fine-tuning. Omitting it reduces nDCG@10 by ~2-4 points.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- Text Splitting ---------------------------------------------------
    CHUNK_SIZE: int    = 512    # characters; balanced for dense retrieval
    CHUNK_OVERLAP: int = 64     # cross-boundary context preservation

    # --- BGE Bi-Encoder (dense retrieval) ---------------------------------
    # HuggingFaceBgeEmbeddings handles the query instruction prefix internally
    # when we set query_instruction. For documents, instruction must be "".
    BIENCODER_MODEL: str          = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE: str         = "cpu"           # "cuda" for GPU
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- BGE Cross-Encoder (reranking) ------------------------------------
    CROSSENCODER_MODEL: str = "BAAI/bge-reranker-base"

    # --- FAISS Index Configuration ----------------------------------------
    # IndexFlatIP = Inner Product index (exact search, no quantization)
    # For production with 10M+ vectors, use IndexIVFPQ for approximate search
    FAISS_PERSIST_DIR: str = "./faiss_index"

    # --- Retrieval Topology -----------------------------------------------
    BIENCODER_CANDIDATE_K: int = 20    # bi-encoder fetches this many candidates
    BM25_CANDIDATE_K: int      = 20    # BM25 fetches this many candidates
    RERANKER_TOP_N: int        = 4     # cross-encoder returns top N after reranking
    ENSEMBLE_WEIGHTS: list     = [0.4, 0.6]  # [BM25 weight, Dense weight]

    # --- Ollama LLM -------------------------------------------------------
    # Requires Ollama to be running locally: `ollama serve`
    # Model must be pulled first: `ollama pull qwen2.5-coder:7b`
    OLLAMA_BASE_URL: str   = "http://localhost:11434"
    OLLAMA_MODEL: str      = "qwen2.5-coder:7b"
    LLM_TEMPERATURE: float = 0.0
    LLM_MAX_TOKENS: int    = 1024

    # --- RAG Generation Prompt -------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information provided in the context below.
If the answer is not present in the context, respond with:
"The provided documents do not contain sufficient information to answer this question."

Context (retrieved from research papers):
{context}

Question: {question}

Provide a technically accurate, well-structured answer:"""



In [7]:


# ===========================================================================
# SECTION 2 -- PDF DOWNLOADER
# ===========================================================================

def download_pdfs(config: DenseRAGConfig) -> List[Path]:
    """
    Download research PDFs from arXiv with file-existence caching.

    Args:
        config: DenseRAGConfig with PDF_SOURCES and PDF_DOWNLOAD_DIR.

    Returns:
        List of Path objects for downloaded PDFs.

    Raises:
        RuntimeError: If download fails or produces undersized file.
    """
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue

        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-Research-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small ({len(data)} bytes): {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(
                f"Cannot download '{url}'. Place file manually at '{dest}'."
            ) from exc

    return paths


In [8]:

# ===========================================================================
# SECTION 3 -- PDF LOADING AND CHUNKING
# ===========================================================================

def load_and_chunk_documents(
    pdf_paths: List[Path],
    config: DenseRAGConfig,
) -> List[Document]:
    """
    Load PDF pages via PyPDFLoader and chunk with RecursiveCharacterTextSplitter.

    For dense retrieval, chunk size has a direct effect on embedding quality.
    Too small (< 100 chars): the embedding vector captures only a fragment,
    lacking enough semantic signal. Too large (> 1000 chars): the vector
    becomes a centroid across multiple topics, reducing retrieval precision.
    512 characters (roughly 80-100 words) is the empirically validated
    sweet spot for research paper Q&A with BGE models.

    Args:
        pdf_paths : Downloaded PDF Path objects.
        config    : DenseRAGConfig with CHUNK_SIZE and CHUNK_OVERLAP.

    Returns:
        List of chunked Documents annotated with paper_name metadata.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,
    )

    all_chunks: List[Document] = []

    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        logger.info("Loading: %s", pdf_path.name)

        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        for page in pages:
            page.metadata["source"]     = pdf_path.name
            page.metadata["paper_name"] = paper_name

        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)

    logger.info(
        "Total chunks: %d  avg %.0f chars/chunk",
        len(all_chunks),
        sum(len(c.page_content) for c in all_chunks) / max(len(all_chunks), 1),
    )
    return all_chunks


In [18]:

# ===========================================================================
# SECTION 4 -- BI-ENCODER EMBEDDING MODEL (BGE)
# ===========================================================================
from langchain_core.embeddings import Embeddings as LCEmbeddings

# Module-level cache: the heavy model (~1.3 GB) is loaded only once per
# kernel session, even if main() or build_bge_embeddings() is called multiple
# times (e.g. after changing a config value or re-running a single cell).
_GLOBAL_EMBEDDINGS: Optional[HuggingFaceEmbeddings] = None


class BGEQueryEmbeddings(LCEmbeddings):
    """
    Wrapper around HuggingFaceEmbeddings that prepends the BGE query
    instruction to embed_query() calls only.

    BGE models require the instruction prefix ONLY for queries, NOT for
    documents.  embed_documents() is delegated unchanged to the base model.

    Inherits from langchain_core.embeddings.Embeddings (an ABC) so that
    FAISS and other LangChain components that use isinstance(..., Embeddings)
    recognise this object as a proper embeddings provider.

    Using a plain Python class (not a Pydantic subclass) sidesteps the
    Pydantic v2 restriction that prevents setting instance attributes that
    are not declared as model fields.
    """

    def __init__(self, base: HuggingFaceEmbeddings, instruction: str) -> None:
        self._base = base
        self._instruction = instruction

    def embed_query(self, text: str) -> List[float]:
        """Prepend BGE query instruction, then delegate to base model."""
        return self._base.embed_query(self._instruction + text)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """No instruction prefix for documents -- delegate directly."""
        return self._base.embed_documents(texts)


def build_bge_embeddings(config: DenseRAGConfig) -> BGEQueryEmbeddings:
    """
    Return (or reuse) the BAAI/bge-large-en-v1.5 bi-encoder embedding model.

    The underlying HuggingFaceEmbeddings instance is cached in the module-level
    _GLOBAL_EMBEDDINGS variable so the model is loaded at most once per kernel
    session, regardless of how many times main() is called.

    Returns a BGEQueryEmbeddings wrapper that prepends the query instruction
    in embed_query() while leaving embed_documents() unmodified.

    Args:
        config: DenseRAGConfig with BIENCODER_MODEL, BIENCODER_DEVICE,
                and BIENCODER_QUERY_INSTRUCTION.

    Returns:
        BGEQueryEmbeddings instance compatible with FAISS and LangChain retrievers.
    """
    global _GLOBAL_EMBEDDINGS

    if _GLOBAL_EMBEDDINGS is None:
        logger.info(
            "Loading BGE bi-encoder: %s (device=%s) -- may download ~1.3 GB on first run ...",
            config.BIENCODER_MODEL, config.BIENCODER_DEVICE,
        )
        _GLOBAL_EMBEDDINGS = HuggingFaceEmbeddings(
            model_name=config.BIENCODER_MODEL,
            model_kwargs={"device": config.BIENCODER_DEVICE},
            encode_kwargs={"normalize_embeddings": True},
        )
        logger.info("BGE bi-encoder loaded. Embedding dimension: 1024")
    else:
        logger.info("Reusing cached BGE bi-encoder (already in memory).")

    return BGEQueryEmbeddings(_GLOBAL_EMBEDDINGS, config.BIENCODER_QUERY_INSTRUCTION)


In [10]:

# ===========================================================================
# SECTION 5 -- CROSS-ENCODER (BGE RERANKER)
# ===========================================================================

def build_cross_encoder(config: DenseRAGConfig) -> HuggingFaceCrossEncoder:
    """
    Initialize the BAAI/bge-reranker-base cross-encoder for reranking.

    The cross-encoder takes (query, document) pairs as input and outputs
    a scalar logit score (higher = more relevant). It processes both texts
    jointly through a full transformer attention stack, enabling token-level
    interactions between query and document tokens.

    This is fundamentally different from the bi-encoder, which compresses
    each text to a single vector before any comparison can occur. The
    cross-encoder's joint attention means it can notice things like:
        Query: "What is the learning rate used in BERT pre-training?"
        Document: "...We train with a peak learning rate of 1e-4..."
    The cross-encoder directly attends to "learning rate" in the query
    while processing "peak learning rate of 1e-4" in the document --
    producing a high relevance score even if the phrasing differs.

    Model choice rationale (bge-reranker-base vs. alternatives):
        - bge-reranker-base  : BERT-base (110M params), good quality/speed trade-off
        - bge-reranker-large : BERT-large (330M params), higher accuracy, 2.5x slower
        - ms-marco-MiniLM-L-6-v2: 6-layer MiniLM, fastest, lower quality
        - cross-encoder/nli-deberta-v3-base: good for NLI-style queries
    For research paper Q&A, bge-reranker-base gives the best balance.

    Args:
        config: DenseRAGConfig with CROSSENCODER_MODEL.

    Returns:
        Initialized HuggingFaceCrossEncoder instance.
    """
    logger.info("Loading cross-encoder: %s", config.CROSSENCODER_MODEL)
    cross_encoder = HuggingFaceCrossEncoder(model_name=config.CROSSENCODER_MODEL)
    logger.info("Cross-encoder loaded.")
    return cross_encoder


In [11]:


# ===========================================================================
# SECTION 6 -- OLLAMA LLM
# ===========================================================================

def build_ollama_llm(config: DenseRAGConfig) -> ChatOllama:
    """
    Initialize the Ollama-backed local LLM (qwen2.5-coder:7b).

    Ollama runs a local inference server that serves open-source models
    via an OpenAI-compatible REST API. No API keys or internet access are
    required at inference time -- the model weights are stored locally.

    Prerequisites:
        1. Ollama installed: https://ollama.com/download
        2. Ollama server running: `ollama serve`
        3. Model downloaded: `ollama pull qwen2.5-coder:7b`

    qwen2.5-coder:7b is a 7B-parameter model from Alibaba's Qwen2.5 series,
    fine-tuned for code and technical reasoning tasks. It performs well on
    research paper Q&A while running fully locally.

    num_predict controls the maximum number of tokens generated.
    temperature=0.0 ensures deterministic, factual responses.

    Args:
        config: DenseRAGConfig with OLLAMA_BASE_URL, OLLAMA_MODEL,
                LLM_TEMPERATURE, and LLM_MAX_TOKENS.

    Returns:
        Initialized ChatOllama instance.

    Raises:
        ConnectionError: If the Ollama server is not reachable.
    """
    import urllib.request as _req

    # Validate that the Ollama server is reachable before proceeding
    try:
        _req.urlopen(f"{config.OLLAMA_BASE_URL}/api/tags", timeout=5)
    except Exception as exc:
        raise ConnectionError(
            f"Cannot reach Ollama server at '{config.OLLAMA_BASE_URL}'.\n"
            "  1. Install Ollama: https://ollama.com/download\n"
            "  2. Start the server: ollama serve\n"
            "  3. Pull the model:   ollama pull qwen2.5-coder:7b"
        ) from exc

    logger.info(
        "Ollama LLM: model='%s'  base_url='%s'",
        config.OLLAMA_MODEL, config.OLLAMA_BASE_URL,
    )
    return ChatOllama(
        model=config.OLLAMA_MODEL,
        base_url=config.OLLAMA_BASE_URL,
        temperature=config.LLM_TEMPERATURE,
        num_predict=config.LLM_MAX_TOKENS,
    )



In [12]:

# ===========================================================================
# SECTION 7 -- FAISS VECTOR STORE BUILDER
# ===========================================================================

def build_faiss_index(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    config: DenseRAGConfig,
    force_rebuild: bool = False,
) -> FAISS:
    """
    Build or load a FAISS vector index from document chunks.

    FAISS (Facebook AI Similarity Search) is the industry-standard library
    for exact and approximate nearest-neighbor search on dense vectors.
    It is used here with IndexFlatIP (exact inner product search) because:
        1. With normalize_embeddings=True, inner product = cosine similarity.
        2. For a few thousand research paper chunks, exact search (O(N*D)) is
           fast enough -- IndexFlatIP on 5000 * 1024-dim vectors takes ~1ms.
        3. For 1M+ vectors, replace with IndexIVFPQ (approximate, 100x faster)
           by setting the nlist and nprobe FAISS parameters.

    FAISS.save_local() persists the index to disk as two files:
        faiss_index/index.faiss  -- the binary ANN index
        faiss_index/index.pkl    -- the document store (docstore + index_to_docstore_id)

    FAISS.load_local() restores both files, enabling warm starts where the
    expensive embedding step is skipped entirely.

    allow_dangerous_deserialization=True is required by LangChain 0.2+ when
    loading a FAISS index because the .pkl file uses Python pickle, which can
    execute arbitrary code if tampered with. In a controlled environment
    where you generated the index yourself, this is safe.

    Args:
        chunks        : Document chunks to embed and index.
        embeddings    : BGE bi-encoder embedding model.
        config        : DenseRAGConfig with FAISS_PERSIST_DIR.
        force_rebuild : If True, delete existing index and rebuild.

    Returns:
        Populated FAISS vector store instance.
    """
    index_dir = Path(config.FAISS_PERSIST_DIR)
    faiss_file = index_dir / "index.faiss"

    # Load from disk if available and rebuild is not forced
    if faiss_file.exists() and not force_rebuild:
        logger.info("Loading FAISS index from '%s' ...", config.FAISS_PERSIST_DIR)
        try:
            vectorstore = FAISS.load_local(
                folder_path=config.FAISS_PERSIST_DIR,
                embeddings=embeddings,
                allow_dangerous_deserialization=True,
            )
            logger.info(
                "FAISS index loaded. Vectors: %d",
                vectorstore.index.ntotal,
            )
            return vectorstore
        except Exception as exc:
            logger.warning("FAISS load failed (%s), rebuilding ...", exc)

    # Build from scratch
    logger.info("Building FAISS index from %d chunks ...", len(chunks))
    start = time.perf_counter()

    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings,
    )

    elapsed = time.perf_counter() - start
    logger.info(
        "FAISS index built. Vectors: %d  (%.2fs)",
        vectorstore.index.ntotal, elapsed,
    )

    # Persist to disk
    index_dir.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(config.FAISS_PERSIST_DIR)
    logger.info("FAISS index persisted to '%s'", config.FAISS_PERSIST_DIR)

    return vectorstore



In [13]:

# ===========================================================================
# SECTION 8 -- FIVE RETRIEVAL CONFIGURATIONS
# Each configuration builds on the previous, demonstrating the quality
# improvement at each stage of the dense retrieval pipeline.
# ===========================================================================

def build_config1_bm25(
    chunks: List[Document],
    config: DenseRAGConfig,
) -> BM25Retriever:
    """
    Configuration 1: Pure BM25 Sparse Retrieval (baseline).

    BM25 (Best Match 25) is a probabilistic ranking function based on
    term frequency (TF) and inverse document frequency (IDF) with document
    length normalization. It has no semantic understanding -- it matches
    tokens exactly. Used here as the BASELINE to quantify how much dense
    retrieval improves over lexical matching.

    BM25Retriever operates entirely in-memory using the rank-bm25 library.
    It is stateless between queries -- the full BM25 scoring over all
    documents runs at query time (O(N) but extremely fast for <100K docs).

    BM25 parameters (rank-bm25 defaults used here):
        k1 = 1.5   -- term frequency saturation (higher = more saturation)
        b  = 0.75  -- document length normalization (0=none, 1=full)
    These are the Okapi BM25 standard defaults validated across TREC benchmarks.

    Args:
        chunks : All document chunks.
        config : DenseRAGConfig with BM25_CANDIDATE_K.

    Returns:
        Initialized BM25Retriever.
    """
    logger.info("Building BM25 retriever from %d chunks ...", len(chunks))
    retriever = BM25Retriever.from_documents(chunks)
    retriever.k = config.BM25_CANDIDATE_K
    logger.info("BM25 retriever ready.")
    return retriever


def build_config2_dense_biencoder(
    vectorstore: FAISS,
    config: DenseRAGConfig,
):
    """
    Configuration 2: Pure Dense Bi-Encoder Retrieval.

    Encodes the query using BGE-large-en-v1.5 (with query instruction prefix)
    and performs ANN search in the FAISS index using inner-product similarity.
    Returns the top BIENCODER_CANDIDATE_K most similar chunks.

    This is the pure dense retrieval baseline, demonstrating semantic recall
    without any reranking. The quality gap between Config 2 and Config 3
    (which adds cross-encoder reranking) quantifies the precision improvement
    that reranking provides.

    Args:
        vectorstore : Populated FAISS index.
        config      : DenseRAGConfig with BIENCODER_CANDIDATE_K.

    Returns:
        FAISS-backed retriever.
    """
    logger.info("Building dense bi-encoder retriever (k=%d)...", config.BIENCODER_CANDIDATE_K)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.BIENCODER_CANDIDATE_K},
    )
    return retriever


def build_config3_dense_with_reranker(
    vectorstore: FAISS,
    cross_encoder: HuggingFaceCrossEncoder,
    config: DenseRAGConfig,
) -> ContextualCompressionRetriever:
    """
    Configuration 3: Dense Bi-Encoder + Cross-Encoder Reranker.

    This is the core two-stage dense retrieval architecture:

    Stage 1 (Recall): FAISS bi-encoder retrieves top-20 semantic candidates.
        Fast: 1 embedding call + ANN search. Milliseconds.
        Goal: HIGH RECALL -- ensure the relevant document is in the top-20.

    Stage 2 (Precision): Cross-encoder re-scores all 20 candidates jointly
        with the query. Slow but applied only to 20 docs, not the full corpus.
        CrossEncoderReranker calls cross_encoder.predict([(q, d1), ..., (q, d20)])
        in a single batch, producing 20 relevance scores.
        Documents are sorted by score. Top RERANKER_TOP_N are returned.
        Goal: HIGH PRECISION -- ensure the top-4 are the truly relevant ones.

    ContextualCompressionRetriever is LangChain's general wrapper for
    "retrieve then compress/filter/rerank" patterns. The `base_retriever`
    handles the first stage (recall) and the `base_compressor` handles the
    second stage (precision). CrossEncoderReranker is the compressor here.

    Latency profile (typical):
        Bi-encoder query embedding : ~5ms  (CPU, bge-large)
        FAISS ANN search (5000 vecs): ~1ms
        Cross-encoder reranking (20 pairs): ~80ms  (CPU, bge-reranker-base)
        Total query latency : ~90ms on CPU

    Args:
        vectorstore    : Populated FAISS index.
        cross_encoder  : Initialized HuggingFaceCrossEncoder.
        config         : DenseRAGConfig.

    Returns:
        ContextualCompressionRetriever (bi-encoder + cross-encoder).
    """
    logger.info(
        "Building Dense + CrossEncoder retriever (candidates=%d, top_n=%d)...",
        config.BIENCODER_CANDIDATE_K, config.RERANKER_TOP_N,
    )

    # Stage 1: bi-encoder fetches candidates
    base_retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.BIENCODER_CANDIDATE_K},
    )

    # Stage 2: cross-encoder reranks candidates
    compressor = CrossEncoderReranker(
        model=cross_encoder,
        top_n=config.RERANKER_TOP_N,
    )

    retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=base_retriever,
    )
    return retriever


def build_config4_hybrid(
    vectorstore: FAISS,
    chunks: List[Document],
    config: DenseRAGConfig,
) -> EnsembleRetriever:
    """
    Configuration 4: Hybrid BM25 + Dense Retrieval (no reranking).

    EnsembleRetriever combines results from multiple retrievers using
    Reciprocal Rank Fusion (RRF). RRF assigns each document a fusion score:

        RRF_score(doc) = sum_over_retrievers [ 1 / (k + rank_in_retriever) ]

    where k=60 is the standard constant (from the original RRF paper by
    Cormack et al., 2009). The document that ranks 1st in retriever A and
    5th in retriever B gets a higher RRF score than one that ranks 3rd in
    both, because the top-1 position is weighted much more heavily.

    Why combine BM25 and dense?
        Dense retrieval misses exact keyword matches (zero-shot for rare terms,
        proper nouns, or newly coined acronyms not seen during training).
        BM25 misses semantic paraphrases (sees no overlap in token space).
        RRF fusion ensures that documents which are relevant in EITHER the
        keyword space OR the semantic space are surfaced, while documents
        relevant in BOTH spaces receive the highest combined scores.

    ENSEMBLE_WEIGHTS = [0.4, 0.6]:
        BM25 weight 0.4, Dense weight 0.6.
        Not used by EnsembleRetriever's RRF implementation directly
        (RRF is rank-based, not score-based), but respected in the final
        weighted merge. Tune based on your query distribution.

    Args:
        vectorstore : Populated FAISS index.
        chunks      : All document chunks (for BM25 in-memory index).
        config      : DenseRAGConfig.

    Returns:
        EnsembleRetriever (BM25 + FAISS with RRF fusion).
    """
    logger.info(
        "Building Hybrid (BM25 + Dense) EnsembleRetriever "
        "(bm25_k=%d, dense_k=%d, weights=%s) ...",
        config.BM25_CANDIDATE_K, config.BIENCODER_CANDIDATE_K,
        config.ENSEMBLE_WEIGHTS,
    )

    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = config.BM25_CANDIDATE_K

    dense_retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.BIENCODER_CANDIDATE_K},
    )

    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=config.ENSEMBLE_WEIGHTS,
    )
    logger.info("EnsembleRetriever ready (RRF fusion).")
    return ensemble_retriever


def build_config5_hybrid_with_reranker(
    vectorstore: FAISS,
    chunks: List[Document],
    cross_encoder: HuggingFaceCrossEncoder,
    config: DenseRAGConfig,
) -> ContextualCompressionRetriever:
    """
    Configuration 5: Hybrid BM25 + Dense + Cross-Encoder Reranker.

    This is the PRODUCTION-GRADE GOLD STANDARD pipeline. It combines:
        1. BM25: lexical recall for exact keyword matches
        2. Dense bi-encoder: semantic recall for paraphrases
        3. RRF fusion: merges both lists into a unified ranking
        4. Cross-encoder: precision reranking of the fused top-20 candidates

    This four-stage pipeline represents the best practice from industrial
    retrieval systems (Google, Bing, OpenAI, Cohere all use variants of this).
    The pipeline is designed so that each stage maximizes a different
    objective in sequence:
        BM25 + Dense --> maximize RECALL (find all candidates)
        RRF           --> maximize DIVERSITY (don't miss either signal)
        CrossEncoder  --> maximize PRECISION (return only truly relevant docs)

    The combined retriever output is the top RERANKER_TOP_N documents that
    are both semantically relevant AND lexically matching AND reranked by
    joint query-document attention. In practice, this outperforms any single
    retrieval method by 10-25% on nDCG@10 across BEIR benchmarks.

    Args:
        vectorstore    : Populated FAISS index.
        chunks         : All chunks (for BM25).
        cross_encoder  : Initialized cross-encoder.
        config         : DenseRAGConfig.

    Returns:
        ContextualCompressionRetriever (Hybrid ensemble + CrossEncoder reranker).
    """
    logger.info(
        "Building Full Hybrid + CrossEncoder pipeline "
        "(PRODUCTION STACK) ...",
    )

    # Stage 1+2: BM25 + Dense with RRF fusion
    ensemble_retriever = build_config4_hybrid(vectorstore, chunks, config)

    # Stage 3: Cross-encoder reranks the fused results
    compressor = CrossEncoderReranker(
        model=cross_encoder,
        top_n=config.RERANKER_TOP_N,
    )

    retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=ensemble_retriever,
    )
    logger.info("Config 5 (Hybrid + CrossEncoder) ready.")
    return retriever



In [14]:

# ===========================================================================
# SECTION 9 -- RAG CHAIN FACTORY
# A unified chain builder that wraps any retriever into a complete LCEL chain.
# ===========================================================================

def build_rag_chain(retriever, llm: ChatOllama, config: DenseRAGConfig):
    """
    Build a complete LCEL RAG chain wrapping any retriever configuration.

    Because all five configurations expose a standard LangChain Runnable
    retriever interface (.invoke(query) -> List[Document]), this single
    factory function produces a valid RAG chain for all of them.

    Context formatting includes paper name, page, and source offset for
    provenance tracking, and the character length of each chunk to help
    identify retrieval issues (very short chunks may indicate BM25 fragments).

    Args:
        retriever : Any of the five configured retrievers.
        llm       : ChatOllama instance (local Ollama-backed LLM).
        config    : DenseRAGConfig with RAG_PROMPT.

    Returns:
        Tuple of (LCEL rag_chain, retriever).
    """
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)

    def format_docs(docs: List[Document]) -> str:
        """
        Format retrieved documents into an annotated context block.
        Each chunk is labeled with paper name, page, character count,
        and cross-encoder score if available in metadata.
        """
        parts = []
        for i, doc in enumerate(docs, start=1):
            paper    = doc.metadata.get("paper_name", "Unknown")
            page     = doc.metadata.get("page", "?")
            chars    = len(doc.page_content)
            # CrossEncoderReranker injects 'relevance_score' into metadata
            score    = doc.metadata.get("relevance_score", None)
            score_str = f" | rerank_score={score:.4f}" if score is not None else ""
            header = f"[Chunk {i} | {paper} | Page {page} | {chars} chars{score_str}]"
            parts.append(f"{header}\n{doc.page_content.strip()}")
        return ("\n\n" + "-" * 60 + "\n\n").join(parts)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )
    return chain, retriever



In [15]:

# ===========================================================================
# SECTION 10 -- LATENCY PROFILER AND QUERY INTERFACE
# ===========================================================================

def timed_retrieve(retriever, query: str) -> Tuple[List[Document], float]:
    """
    Execute retrieval and measure wall-clock latency in milliseconds.

    Timing dense retrieval accurately requires care: the first call to any
    PyTorch model is slower due to JIT compilation and memory allocation.
    We do NOT warm up the models in this demo, so the first query of each
    configuration will be slightly slower than subsequent ones.

    In production, implement a warmup call at startup with a representative
    query to eliminate this cold-start overhead from production latency SLOs.

    Args:
        retriever : Any LangChain retriever.
        query     : The natural language query string.

    Returns:
        Tuple of (list of retrieved Documents, latency in milliseconds).
    """
    start = time.perf_counter()
    docs  = retriever.invoke(query)
    elapsed_ms = (time.perf_counter() - start) * 1000
    return docs, elapsed_ms


def query_all_configs(
    config_chains: Dict[str, tuple],
    question: str,
    show_retrieval_details: bool = True,
) -> Dict[str, Dict]:
    """
    Run a single question through all five retrieval configurations.

    Prints per-configuration retrieval statistics (candidate count, latency,
    cross-encoder scores) and the LLM-generated answer. Also generates a
    latency and quality comparison summary table at the end.

    Args:
        config_chains          : Dict mapping config label to (chain, retriever).
        question               : Natural language question.
        show_retrieval_details : If True, print retrieved chunk metadata.

    Returns:
        Dict mapping config label to {"answer": str, "latency_ms": float,
        "n_docs": int}.
    """
    results: Dict[str, Dict] = {}

    print("\n" + "#" * 76)
    print(f"QUESTION: {question}")
    print("#" * 76)

    for label, (chain, retriever) in config_chains.items():
        print(f"\n{'=' * 60}")
        print(f"CONFIG: {label}")
        print("=" * 60)

        # Retrieval with timing
        docs, retrieval_ms = timed_retrieve(retriever, question)

        if show_retrieval_details:
            print(f"\nRetrieved {len(docs)} chunks  (latency: {retrieval_ms:.1f}ms)")
            for i, doc in enumerate(docs, 1):
                paper = doc.metadata.get("paper_name", "?")
                page  = doc.metadata.get("page", "?")
                score = doc.metadata.get("relevance_score", None)
                chars = len(doc.page_content)
                score_str = f"  rerank={score:.4f}" if score else ""
                snip  = doc.page_content[:200].replace("\n", " ").strip()
                print(f"  [{i}] {paper} | Page {page} | {chars} chars{score_str}")
                print(f"       {snip}...")

        # Answer generation
        gen_start  = time.perf_counter()
        answer     = chain.invoke(question)
        gen_ms     = (time.perf_counter() - gen_start) * 1000

        print(f"\nANSWER (gen: {gen_ms:.0f}ms):\n{answer}")

        results[label] = {
            "answer":         answer,
            "retrieval_ms":   round(retrieval_ms, 1),
            "generation_ms":  round(gen_ms, 1),
            "n_docs":         len(docs),
        }

    # Comparison summary table
    print("\n" + "=" * 76)
    print("RETRIEVAL LATENCY SUMMARY")
    print(f"{'Configuration':<45} {'Docs':>5} {'Retr(ms)':>10} {'Gen(ms)':>10}")
    print("-" * 76)
    for label, r in results.items():
        print(
            f"{label:<45} {r['n_docs']:>5} "
            f"{r['retrieval_ms']:>10.1f} {r['generation_ms']:>10.0f}"
        )
    print("=" * 76 + "\n")

    return results



In [16]:

# ===========================================================================
# SECTION 11 -- MAIN ORCHESTRATOR
# ===========================================================================

def main() -> None:
    """
    End-to-end Dense Retrieval RAG pipeline orchestrator.

    Execution order:
        1.  Download three arXiv PDFs (cached after first run).
        2.  Load and chunk documents (512 chars, 64 overlap).
        3.  Connect to local Ollama server (fail-fast if not running).
        4.  Load BGE bi-encoder (BAAI/bge-large-en-v1.5, ~1.3GB download).
        5.  Load BGE cross-encoder (BAAI/bge-reranker-base, ~440MB download).
        6.  Build / load FAISS index (persisted to disk, skip re-embedding).
        7.  Assemble all five retrieval configurations.
        8.  Build LCEL RAG chains for each configuration.
        9.  Run comparative demo queries with latency profiling.

    Model download sizes (first run only, cached by HuggingFace Hub):
        BGE-large-en-v1.5 : ~1.3 GB
        BGE-reranker-base  : ~440 MB
    Total: ~1.7 GB download on first run. Subsequent runs use local cache.

    FAISS index: Built on first run (~5-15 sec for 3 papers), persisted to
    ./faiss_index/. Loaded in <1 sec on subsequent runs.

    Ollama LLM: qwen2.5-coder:7b runs locally via Ollama.
    Ensure Ollama is running (`ollama serve`) and the model is pulled
    (`ollama pull qwen2.5-coder:7b`) before executing this notebook.
    """
    config = DenseRAGConfig()
    logger.info("=== Dense Retrieval RAG Pipeline Starting ===")

    # Step 1-2: Data
    pdf_paths = download_pdfs(config)
    chunks    = load_and_chunk_documents(pdf_paths, config)

    # Step 3: LLM (connects to local Ollama server, validates connectivity)
    llm = build_ollama_llm(config)

    # Step 4: BGE bi-encoder
    embeddings = build_bge_embeddings(config)

    # Step 5: BGE cross-encoder
    cross_encoder = build_cross_encoder(config)

    # Step 6: FAISS index (cached after first run)
    vectorstore = build_faiss_index(chunks, embeddings, config)

    # Step 7: Build all five retrieval configurations
    logger.info("Assembling all five retrieval configurations ...")

    r1_bm25           = build_config1_bm25(chunks, config)
    r2_dense          = build_config2_dense_biencoder(vectorstore, config)
    r3_dense_rerank   = build_config3_dense_with_reranker(vectorstore, cross_encoder, config)
    r4_hybrid         = build_config4_hybrid(vectorstore, chunks, config)
    r5_hybrid_rerank  = build_config5_hybrid_with_reranker(vectorstore, chunks, cross_encoder, config)

    # Step 8: Build LCEL RAG chains
    chain1, _ = build_rag_chain(r1_bm25,          llm, config)
    chain2, _ = build_rag_chain(r2_dense,          llm, config)
    chain3, _ = build_rag_chain(r3_dense_rerank,   llm, config)
    chain4, _ = build_rag_chain(r4_hybrid,         llm, config)
    chain5, _ = build_rag_chain(r5_hybrid_rerank,  llm, config)

    config_chains = {
        "Config 1: BM25 (sparse baseline)":      (chain1, r1_bm25),
        "Config 2: Dense Bi-Encoder (FAISS)":     (chain2, r2_dense),
        "Config 3: Dense + Cross-Encoder":        (chain3, r3_dense_rerank),
        "Config 4: Hybrid BM25+Dense (RRF)":      (chain4, r4_hybrid),
        "Config 5: Hybrid + CrossEncoder [BEST]": (chain5, r5_hybrid_rerank),
    }

    # Step 9: Demo queries -- each designed to stress-test a specific
    # aspect of dense vs. sparse vs. reranking capability
    demo_questions = [
        # Tests vocabulary mismatch handling (dense excels, BM25 fails)
        "How does the self-attention mechanism enable the model to capture long-range dependencies?",

        # Tests exact term recall (BM25 excels, dense may miss abbreviations)
        "What is the BLEU score achieved by the Transformer on WMT 2014 English-to-German translation?",

        # Tests multi-sentence reasoning (cross-encoder excels)
        "How does BERT's next sentence prediction objective complement masked language modeling during pre-training?",

        # Tests cross-paper semantic linking (hybrid excels)
        "What retrieval and generation components are combined in the RAG architecture and how do they interact?",

        # Tests precise factual extraction (reranking precision matters most)
        "What are the specific hyperparameter values used for the base Transformer model architecture?",
    ]

    logger.info("Running %d demo queries across all 5 configurations ...", len(demo_questions))

    for question in demo_questions:
        query_all_configs(config_chains, question, show_retrieval_details=True)

    logger.info("=== Dense Retrieval RAG Pipeline Demo Complete ===")


In [19]:
main()

2026-05-13 22:22:19 | INFO     | dense_retrieval_rag | === Dense Retrieval RAG Pipeline Starting ===
2026-05-13 22:22:19 | INFO     | dense_retrieval_rag | Cached: attention_is_all_you_need.pdf (2163.3 KB)
2026-05-13 22:22:19 | INFO     | dense_retrieval_rag | Cached: bert_pretraining_transformers.pdf (757.0 KB)
2026-05-13 22:22:19 | INFO     | dense_retrieval_rag | Cached: rag_knowledge_intensive_nlp.pdf (864.6 KB)
2026-05-13 22:22:19 | INFO     | dense_retrieval_rag | Loading: attention_is_all_you_need.pdf
2026-05-13 22:22:22 | INFO     | dense_retrieval_rag |   attention_is_all_you_need.pdf -> 15 pages -> 92 chunks
2026-05-13 22:22:22 | INFO     | dense_retrieval_rag | Loading: bert_pretraining_transformers.pdf
2026-05-13 22:22:30 | INFO     | dense_retrieval_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 151 chunks
2026-05-13 22:22:30 | INFO     | dense_retrieval_rag | Loading: rag_knowledge_intensive_nlp.pdf
2026-05-13 22:22:32 | INFO     | dense_retrieval_rag |   rag_kn

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1791.95it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-13 22:22:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 22:22:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-13 22:22:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 22:22:46 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-13 22:22:46 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-13 22:22:47 | INFO     | httpx |

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2127.48it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-13 22:22:51 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 22:22:52 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-05-13 22:22:52 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 22:22:52 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-13 22:22:53 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-13 22:22:53 | INFO     | httpx |

...

2026-05-13 22:42:45 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

ANSWER (gen: 26603ms):
The specific hyperparameter values used for the base Transformer model architecture are as follows:

- Number of layers (L): 6
- Hidden size (H): 512
- Number of self-attention heads (A): 8
- Dropout probability (Pdrop): 0.1

CONFIG: Config 4: Hybrid BM25+Dense (RRF)

Retrieved 34 chunks  (latency: 234.1ms)
  [1] Bert Pretraining Transformers | Page 12 | 468 chars
       positional embeddings. A.3 Fine-tuning Procedure For ﬁne-tuning, most model hyperparameters are the same as in pre-training, with the exception of the batch size, learning rate, and number of train- i...
  [2] Attention Is All You Need | Page 8 | 495 chars
       Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Liste...
  [3] Attention Is All Y

Batches: 100%|██████████| 2/2 [00:14<00:00,  7.48s/it]



Retrieved 4 chunks  (latency: 15219.5ms)
  [1] Attention Is All You Need | Page 7 | 463 chars
       were written at 10-minute intervals. For the big models, we averaged the last 20 checkpoints. We used beam search with a beam size of 4 and length penalty α = 0 .6 [38]. These hyperparameters were cho...
  [2] Attention Is All You Need | Page 3 | 430 chars
       much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms perform similarly, addi...
  [3] Attention Is All You Need | Page 8 | 495 chars
       Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Liste...
  [4] Bert Pretraining Transformers | Page 2 | 471 chars
       plementation is almost identical to the original, we will omit an exhaustive background descrip- tion 

Batches: 100%|██████████| 2/2 [00:11<00:00,  5.69s/it]


2026-05-13 22:44:50 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

ANSWER (gen: 19484ms):
The provided documents do not contain sufficient information to answer this question.

RETRIEVAL LATENCY SUMMARY
Configuration                                  Docs   Retr(ms)    Gen(ms)
----------------------------------------------------------------------------
Config 1: BM25 (sparse baseline)                 20        5.8       8012
Config 2: Dense Bi-Encoder (FAISS)               20      273.2      26418
Config 3: Dense + Cross-Encoder                   4    10899.2      26602
Config 4: Hybrid BM25+Dense (RRF)                34      234.1      80108
Config 5: Hybrid + CrossEncoder [BEST]            4    15219.5      19484

2026-05-13 22:44:53 | INFO     | dense_retrieval_rag | === Dense Retrieval RAG Pipeline Demo Complete ===
